In [1]:
from collections.abc import Callable
from functools import partial
from typing import Literal

import equinox as eqx
import grain
import jax
import jax.numpy as jnp
import optax
from context_flux_no.data import TheWellDataSource
from context_flux_no.models.fno import FNO
from context_flux_no.nn.operators.fourier_utils import append_grid_channels
from context_flux_no.training.trainer import Trainer
from context_flux_no.utils import num_parameters
from einops import pack, unpack
from equinox._misc import default_floating_dtype
from jaxtyping import Array, Float, PRNGKeyArray


jax.config.update("jax_default_device", jax.devices("gpu")[2])

In [2]:
class ResidualMLP(eqx.Module):
    layers: tuple[eqx.nn.Linear, ...]
    activation: Callable
    final_activation: Callable
    use_bias: bool = eqx.field(static=True)
    use_final_bias: bool = eqx.field(static=True)
    in_size: int | Literal["scalar"] = eqx.field(static=True)
    out_size: int | Literal["scalar"] = eqx.field(static=True)
    width_size: int = eqx.field(static=True)
    depth: int = eqx.field(static=True)

    def __init__(
        self,
        in_size: int | Literal["scalar"],
        out_size: int | Literal["scalar"],
        width_size: int,
        depth: int,
        activation: Callable = jax.nn.gelu,
        final_activation: Callable = lambda x: x,
        use_bias: bool = True,
        use_final_bias: bool = True,
        dtype=None,
        *,
        scan: bool = False,
        key: PRNGKeyArray,
    ):
        layers = []
        dtype = default_floating_dtype() if dtype is None else dtype
        keys = jax.random.split(key, depth + 1)

        if depth == 0:
            layers.append(
                eqx.nn.Linear(
                    in_size, out_size, use_bias=use_final_bias, dtype=dtype, key=keys[0]
                )
            )
        else:
            layers.append(
                eqx.nn.Linear(in_size, width_size, use_bias, dtype=dtype, key=keys[0])
            )
            layers.extend(
                eqx.nn.Linear(
                    width_size, width_size, use_bias, dtype=dtype, key=keys[i]
                )
                for i in range(1, depth)
            )
            layers.append(
                eqx.nn.Linear(
                    width_size, out_size, use_final_bias, dtype=dtype, key=keys[-1]
                )
            )

        self.layers = tuple(layers)
        self.in_size = in_size
        self.out_size = out_size
        self.width_size = width_size
        self.depth = depth
        self.activation = activation
        self.final_activation = final_activation
        self.use_bias = use_bias
        self.use_final_bias = use_final_bias

    def __call__(
        self, x: Float[Array, " in_size"], *, key: PRNGKeyArray | None = None
    ) -> Float[Array, " out_size"]:
        del key

        if self.depth == 0:
            [layer] = self.layers
            return layer(x)

        else:
            x = self.layers[0](x)
            for layer in self.layers[1:-1]:
                x = x + self.activation(layer(x))
            return self.final_activation(self.layers[-1](x))


In [3]:
class FluxModel(eqx.Module):
    in_channels: int = eqx.field(static=True)
    out_channels: int = eqx.field(static=True)
    stencil_widths: tuple[int, int] = eqx.field(static=True)
    lift_dim: int = eqx.field(static=True)
    hidden_dim: int = eqx.field(static=True)
    depth: int = eqx.field(static=True)
    activation: Callable

    lift_layer: eqx.nn.Conv1d
    mlp: eqx.nn.MLP

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stencil_widths: tuple[int, int],
        lift_dim: int,
        hidden_dim: int,
        depth: int,
        activation: Callable = jax.nn.gelu,
        dtype=None,
        *,
        key: PRNGKeyArray,
    ):
        keys = jax.random.split(key, 2)
        kernel_size = stencil_widths[0] + stencil_widths[1] + 1
        self.lift_layer = eqx.nn.Conv1d(
            in_channels=in_channels,
            out_channels=lift_dim,
            kernel_size=kernel_size,
            dtype=dtype,
            key=keys[0],
        )
        self.mlp = eqx.nn.MLP(
            in_size=lift_dim,
            out_size=out_channels,
            width_size=hidden_dim,
            depth=depth,
            activation=activation,
            dtype=dtype,
            key=keys[1],
        )
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stencil_widths = stencil_widths
        self.lift_dim = lift_dim
        self.hidden_dim = hidden_dim
        self.depth = depth
        self.activation = activation

    @property
    def stencil_size(self) -> int:
        return sum(self.stencil_widths) + 1

    def __call__(
        self,
        u: Float[Array, "in_channels grids"],
        *,
        key: PRNGKeyArray | None = None,
    ) -> Float[Array, "out_channels grids+1"]:
        a, b = self.stencil_widths
        pad_widths = [(0, 0), (a + 1, b)]
        # Need to change mode if not periodic boundary condition
        u_padded = jnp.pad(u, pad_widths, mode="wrap")
        u_stencils: Float[Array, "lift_dim grids_x+1"] = self.activation(
            self.lift_layer(u_padded)
        )
        f: Float[Array, "out_channels grids_x+1"] = eqx.filter_vmap(
            self.mlp, in_axes=-1, out_axes=-1
        )(u_stencils)
        return f


class FluxNeuralOperator(eqx.Module):
    num_spatial_dims: int
    flux_models: tuple[eqx.Module]

    def __call__(
        self,
        u: Float[Array, " channels *grids"],
        args: tuple[float, float],
        *,
        key: PRNGKeyArray | None = None,
    ):
        dt, *dxs = args
        v: Float[Array, " channels+num_spatial_dims *grids"] = append_grid_channels(u)

        for i, (flux, dx) in enumerate(zip(self.flux_models, dxs)):
            df = self._apply_flux(flux, v, i)
            u = u - dt * df / dx
        return u

    def _apply_flux(
        self, flux, v: Float[Array, " in_channels *grids"], spatial_axis: int
    ) -> Float[Array, " out_channels *grids"]:
        v = jnp.swapaxes(v, spatial_axis + 1, 1)
        v_, ps = pack([v], "C S *")
        f_ = eqx.filter_vmap(flux, in_axes=-1, out_axes=-1)(v_)
        f = unpack(f_, ps, "C S *")[0]
        df = jnp.diff(f, axis=1)
        df = jnp.swapaxes(df, spatial_axis + 1, 1)
        return df


class FluxNeuralOperatorv2(eqx.Module):
    num_spatial_dims: int = eqx.field(static=True)
    lift_dim: int = eqx.field(static=True)
    stack_grid: bool = eqx.field(static=True)

    flux_models: tuple[eqx.Module]
    lift_layer: eqx.nn.Sequential
    project_layer: eqx.nn.Sequential

    def __init__(
        self,
        num_spatial_dims: int,
        channels: int,
        lift_dim: int,
        stencil_widths: tuple[int, int],
        depth: int,
        width_lift: int = 128,
        depth_lift: int = 1,
        width_project: int = 128,
        depth_project: int = 1,
        activation: Callable = jax.nn.gelu,
        stack_grid: bool = True,
        dtype=None,
        *,
        key: PRNGKeyArray,
    ):
        keys = jax.random.split(key, 3)
        Conv1x1 = partial(
            eqx.nn.Conv, num_spatial_dims=num_spatial_dims, kernel_size=1, dtype=dtype
        )
        keys_l = jax.random.split(keys[0], depth_lift + 2)
        lift_layers = [
            Conv1x1(
                in_channels=channels + num_spatial_dims if stack_grid else channels,
                out_channels=width_lift,
                key=keys_l[0],
            ),
            eqx.nn.Lambda(activation),
        ]
        for i in range(depth_lift):
            lift_layers += [
                Conv1x1(
                    in_channels=width_lift, out_channels=width_lift, key=keys_l[i + 1]
                ),
                eqx.nn.Lambda(activation),
            ]
        lift_layers += [
            Conv1x1(in_channels=width_lift, out_channels=lift_dim, key=keys_l[-1])
        ]
        self.lift_layer = eqx.nn.Sequential(lift_layers)

        self.flux_models = [
            FluxModel(
                in_channels=lift_dim,
                out_channels=lift_dim,
                stencil_widths=stencil_widths,
                lift_dim=lift_dim,
                hidden_dim=lift_dim,
                depth=depth,
                key=k,
            )
            for k in jax.random.split(keys[1], num_spatial_dims)
        ]

        keys_p = jax.random.split(keys[0], depth_project + 2)
        project_layers = [
            Conv1x1(in_channels=lift_dim, out_channels=width_project, key=keys_p[0]),
            eqx.nn.Lambda(activation),
        ]
        for i in range(depth_project):
            project_layers += [
                Conv1x1(
                    in_channels=width_project,
                    out_channels=width_project,
                    key=keys_p[i + 1],
                ),
                eqx.nn.Lambda(activation),
            ]
        project_layers += [
            Conv1x1(in_channels=width_project, out_channels=channels, key=keys_l[-1])
        ]
        self.project_layer = eqx.nn.Sequential(project_layers)

        self.num_spatial_dims = num_spatial_dims
        self.lift_dim = lift_dim
        self.stack_grid = stack_grid

    def __call__(
        self,
        u: Float[Array, " channels *grids"],
        args: tuple[float, float],
        *,
        key: PRNGKeyArray | None = None,
    ):
        dt, *dxs = args
        v: Float[Array, " channels+num_spatial_dims *grids"] = (
            append_grid_channels(u) if self.stack_grid else u
        )
        v: Float[Array, " lift_dim *grids"] = self.lift_layer(v)

        for i, (flux, dx) in enumerate(zip(self.flux_models, dxs)):
            df = self._apply_flux(flux, v, i)
            v = v - dt * df / dx
        return self.project_layer(v)

    def _apply_flux(
        self, flux, v: Float[Array, " in_channels *grids"], spatial_axis: int
    ) -> Float[Array, " out_channels *grids"]:
        v = jnp.swapaxes(v, spatial_axis + 1, 1)
        v_, ps = pack([v], "C S *")
        f_ = eqx.filter_vmap(flux, in_axes=-1, out_axes=-1)(v_)
        f = unpack(f_, ps, "C S *")[0]
        df = jnp.diff(f, axis=1)
        df = jnp.swapaxes(df, spatial_axis + 1, 1)
        return df


class FluxModelConv(eqx.Module):
    in_channels: int = eqx.field(static=True)
    out_channels: int = eqx.field(static=True)
    hidden_channels: int = eqx.field(static=True)
    depth: int = eqx.field(static=True)
    activation: Callable

    conv_layers: list[eqx.nn.Conv1d]

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        hidden_channels: int,
        depth: int,
        activation: Callable = jax.nn.gelu,
        dtype=None,
        *,
        key: PRNGKeyArray,
    ):
        keys = jax.random.split(key, depth)

        conv_layers = [
            eqx.nn.Conv1d(
                in_channels=in_channels,
                out_channels=hidden_channels,
                kernel_size=2,
                dtype=dtype,
                key=keys[0],
            )
        ]
        conv_layers.extend(
            eqx.nn.Conv1d(
                in_channels=hidden_channels,
                out_channels=hidden_channels,
                kernel_size=2,
                dtype=dtype,
                key=keys[i],
            )
            for i in range(1, depth)
        )
        conv_layers.append(
            eqx.nn.Conv1d(
                in_channels=hidden_channels,
                out_channels=out_channels,
                kernel_size=2,
                dtype=dtype,
                key=keys[-1],
            )
        )

        self.conv_layers = conv_layers

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.hidden_channels = hidden_channels
        self.depth = depth
        self.activation = activation

    @property
    def stencil_size(self) -> int:
        return sum(self.stencil_widths) + 1

    def __call__(
        self,
        u: Float[Array, "in_channels grids"],
        *,
        key: PRNGKeyArray | None = None,
    ) -> Float[Array, "out_channels grids+1"]:
        pad_widths = [(0, 0), ((self.depth + 2) // 2, (self.depth + 2) // 2)]
        # Need to change mode if not periodic boundary condition
        u_padded = jnp.pad(u, pad_widths, mode="wrap")
        f = u_padded
        for conv in self.conv_layers:
            f = self.activation(conv(f))
        return f


class FluxNeuralOperatorv3(eqx.Module):
    num_spatial_dims: int = eqx.field(static=True)
    lift_dim: int = eqx.field(static=True)
    stack_grid: bool = eqx.field(static=True)

    flux_models: tuple[eqx.Module]
    lift_layer: eqx.nn.Sequential
    project_layer: eqx.nn.Sequential

    def __init__(
        self,
        num_spatial_dims: int,
        channels: int,
        lift_dim: int,
        depth: int,
        width_lift: int = 128,
        depth_lift: int = 1,
        width_project: int = 128,
        depth_project: int = 1,
        activation: Callable = jax.nn.gelu,
        stack_grid: bool = True,
        dtype=None,
        *,
        key: PRNGKeyArray,
    ):
        keys = jax.random.split(key, 3)
        Conv1x1 = partial(
            eqx.nn.Conv, num_spatial_dims=num_spatial_dims, kernel_size=1, dtype=dtype
        )
        keys_l = jax.random.split(keys[0], depth_lift + 2)
        lift_layers = [
            Conv1x1(
                in_channels=channels + num_spatial_dims if stack_grid else channels,
                out_channels=width_lift,
                key=keys_l[0],
            ),
            eqx.nn.Lambda(activation),
        ]
        for i in range(depth_lift):
            lift_layers += [
                Conv1x1(
                    in_channels=width_lift, out_channels=width_lift, key=keys_l[i + 1]
                ),
                eqx.nn.Lambda(activation),
            ]
        lift_layers += [
            Conv1x1(in_channels=width_lift, out_channels=lift_dim, key=keys_l[-1])
        ]
        self.lift_layer = eqx.nn.Sequential(lift_layers)

        self.flux_models = [
            FluxModelConv(
                in_channels=lift_dim,
                out_channels=lift_dim,
                hidden_channels=lift_dim,
                depth=depth,
                key=k,
            )
            for k in jax.random.split(keys[1], num_spatial_dims)
        ]

        keys_p = jax.random.split(keys[0], depth_project + 2)
        project_layers = [
            Conv1x1(in_channels=lift_dim, out_channels=width_project, key=keys_p[0]),
            eqx.nn.Lambda(activation),
        ]
        for i in range(depth_project):
            project_layers += [
                Conv1x1(
                    in_channels=width_project,
                    out_channels=width_project,
                    key=keys_p[i + 1],
                ),
                eqx.nn.Lambda(activation),
            ]
        project_layers += [
            Conv1x1(in_channels=width_project, out_channels=channels, key=keys_l[-1])
        ]
        self.project_layer = eqx.nn.Sequential(project_layers)

        self.num_spatial_dims = num_spatial_dims
        self.lift_dim = lift_dim
        self.stack_grid = stack_grid

    def __call__(
        self,
        u: Float[Array, " channels *grids"],
        args: tuple[float, float],
        *,
        key: PRNGKeyArray | None = None,
    ):
        dt, *dxs = args
        v: Float[Array, " channels+num_spatial_dims *grids"] = (
            append_grid_channels(u) if self.stack_grid else u
        )
        v: Float[Array, " lift_dim *grids"] = self.lift_layer(v)

        for i, (flux, dx) in enumerate(zip(self.flux_models, dxs)):
            df = self._apply_flux(flux, v, i)
            v = v - dt * df / dx
        return self.project_layer(v)

    def _apply_flux(
        self, flux, v: Float[Array, " in_channels *grids"], spatial_axis: int
    ) -> Float[Array, " out_channels *grids"]:
        v = jnp.swapaxes(v, spatial_axis + 1, 1)
        v_, ps = pack([v], "C S *")
        f_ = eqx.filter_vmap(flux, in_axes=-1, out_axes=-1)(v_)
        f = unpack(f_, ps, "C S *")[0]
        df = jnp.diff(f, axis=1)
        df = jnp.swapaxes(df, spatial_axis + 1, 1)
        return df


In [4]:
flux = FluxModelConv(128, 128, 128, 4, key=jax.random.key(0))


In [23]:
flux(jax.random.normal(jax.random.key(0), (128, 100))).shape

(128, 101)

In [4]:
flux_no = FluxNeuralOperator(
    2,
    [
        FluxModel(6, 4, (8, 8), 256, 256, 3, key=jax.random.key(0)),
        FluxModel(6, 4, (8, 8), 256, 256, 3, key=jax.random.key(1)),
    ],
)

In [6]:
flux_no2 = FluxNeuralOperatorv2(
    num_spatial_dims=2,
    channels=5,
    lift_dim=128,
    stencil_widths=(8, 8),
    depth=3,
    key=jax.random.key(0),
)

In [5]:
# flux_no2 = FluxNeuralOperatorv2(
#     num_spatial_dims=2,
#     channels=4,
#     lift_dim=128,
#     stencil_widths=(8, 8),
#     depth=4,
#     key=jax.random.key(0),
# )
flux_no3 = FluxNeuralOperatorv3(
    num_spatial_dims=2,
    channels=5,
    lift_dim=128,
    depth=4,
    key=jax.random.key(0),
)

In [7]:
source = TheWellDataSource(
    "../../data/datasets",
    "euler_multi_quadrants_periodicBC",
    include_filters=["gamma_1.3_CO2"],
    window_size=11,
    downsample_spatial=8,
    # exclude_field_names=["pressure"],
)
sampler = grain.samplers.IndexSampler(len(source), shuffle=True, seed=0)
transforms = [grain.transforms.Batch(batch_size=64)]

dataloader = grain.DataLoader(
    data_source=source,
    sampler=sampler,
    operations=transforms,
    worker_count=24,
    read_options=grain.ReadOptions(num_threads=16, prefetch_buffer_size=50),
)

source_valid = TheWellDataSource(
    "../../data/datasets",
    "euler_multi_quadrants_periodicBC",
    "valid",
    include_filters=["gamma_1.3_CO2"],
    window_size=11,
    downsample_spatial=8,
    # exclude_field_names=["pressure"],
)
sampler_valid = grain.samplers.IndexSampler(len(source_valid), shuffle=True, seed=1)

dataloader_valid = grain.DataLoader(
    data_source=source_valid,
    sampler=sampler_valid,
    operations=transforms,
    worker_count=24,
    read_options=grain.ReadOptions(num_threads=16, prefetch_buffer_size=50),
)


dataiter = iter(dataloader)
batch = next(dataiter)

In [9]:
flux_no2(batch[0, 0], (0.015, 1 / 64, 1 / 64), key=jax.random.key(0)).shape

(5, 64, 64)

In [10]:
num_parameters(flux_no2)

757125

In [32]:
num_parameters(flux_no3)

396420

In [5]:
jnp.mean(batch, axis=(0, 1, 3, 4))

Array([ 0.908344  ,  3.2770276 ,  0.00826597, -0.01598712], dtype=float32)

In [6]:
jnp.std(batch, axis=(0, 1, 3, 4))

Array([0.5452532, 2.8994749, 0.424648 , 0.3790666], dtype=float32)

In [5]:
ds = grain.MapDataset.source(source).seed(seed=0).shuffle().batch(batch_size=64)
performance_config = grain.experimental.pick_performance_config(
    ds=ds, ram_budget_mb=10**4, max_workers=None, max_buffer_size=None
)

In [6]:
performance_config

PerformanceConfig(multiprocessing_options=MultiprocessingOptions(num_workers=64, per_worker_buffer_size=1, enable_profiling=False), read_options=ReadOptions(num_threads=16, prefetch_buffer_size=1000))

In [7]:
batch.nbytes / 10**9

NameError: name 'batch' is not defined

In [15]:
flux_no(batch[0, -2], (0.015, 1 / 64, 1 / 64)).shape

(4, 64, 64)

In [4]:
fno = FNO(
    num_spatial_dims=2,
    in_channels=4,
    lift_dim=128,
    depth=4,
    frequency_modes=10,
    out_channels=4,
    width_lift=128,
    depth_lift=1,
    depth_project=1,
    stack_grid=True,
    residual_connection=True,
    key=jax.random.key(0),
)

In [ ]:
num_parameters(fno)

13207684

In [12]:
def loss_fn(
    model,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, -2], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)
    return jnp.mean((u1 - u1_pred) ** 2), dict()


loss_fn(flux_no2, batch, (0.015, 1 / 64, 1 / 64), jax.random.key(0))

W0604 22:39:28.832232 3837994 hlo_rematerialization.cc:3233] Can't reduce memory use below 60.08GiB (64516449239 bytes) by rematerialization; only reduced to 67.16GiB (72111685664 bytes), down from 67.16GiB (72111685664 bytes) originally
W0604 22:39:39.503191 3837994 bfc_allocator.cc:502] Allocator (GPU_2_bfc) ran out of memory trying to allocate 67.03GiB (rounded to 71975371264)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
W0604 22:39:39.590768 3837994 bfc_allocator.cc:513] ***_________________________________________________________________________________________________


(Array(4.4011893, dtype=float32), {})

In [7]:
def loss_fn_fno(
    model,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    del key, args
    u0, u1 = u[:, -2], u[:, -1]
    # keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(model)(u0)
    return jnp.mean((u1 - u1_pred) ** 2), dict()

In [8]:
loss_fn_fno(fno, batch, None, jax.random.key(0))

(Array(5.3585467, dtype=float32), {})

In [13]:
trainer = Trainer(
    optax.adamw(1e-4),
    loss_fn,  # loss_fn_fno
    "./",
    None,
    {"entity": "jhko725", "project": "prototype_fluxes"},
    config_dict={"model": ""},
)

In [14]:
trainer.train(
    flux_no2,  # fno
    dataloader,
    dataloader_valid,
    (0.015, 1 / 64, 1 / 64),
    num_steps=2000,
    seed=0,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jhko725/.netrc.
wandb: Currently logged in as: jhko725 (jhelab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W0604 22:43:21.968636 3838036 hlo_rematerialization.cc:3233] Can't reduce memory use below 60.08GiB (64516448753 bytes) by rematerialization; only reduced to 67.16GiB (72111685664 bytes), down from 67.16GiB (72111685664 bytes) originally
W0604 22:43:26.270760 3838013 hlo_rematerialization.cc:3233] Can't reduce memory use below 59.96GiB (64381248471 bytes) by rematerialization; only reduced to 130.13GiB (139723866144 bytes), down from 130.13GiB (139723866144 bytes) originally
W0604 22:43:37.055439 3837996 bfc_allocator.cc:502] Allocator (GPU_2_bfc) ran out of memory trying to allocate 67.03GiB (rounded to 71975371264)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
W0604 22:43:37.055788 3837996 bfc_allocator.cc:513] *_*_________________________________________________________________________________________________


Step: 1 | Train loss: 4.40118932723999 | Valid loss: 
            6.533792018890381
Step: 2 | Train loss: 4.626701354980469 | Valid loss: 
            4.681609630584717


Step: 3 | Train loss: 2.633788824081421 | Valid loss: 
            5.870006084442139


Step: 4 | Train loss: 3.737881898880005 | Valid loss: 
            7.361550807952881


...

Step: 1997 | Train loss: 0.01312508899718523 | Valid loss: 
            0.013733676634728909


Step: 1998 | Train loss: 0.015480117872357368 | Valid loss: 
            0.01882319711148739


Step: 1999 | Train loss: 0.019914327189326286 | Valid loss: 
            0.010627739131450653


Step: 2000 | Train loss: 0.015484550967812538 | Valid loss: 
            0.017560603097081184


train_loss,▇█▆▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
valid_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,0.01548
valid_loss,0.01756
